In [ ]:
# ==============================================================================
# PROJET MASTERCAMP 2026 - ÉTAPE 5 : INTERPRÉTATION ET VISUALISATION
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration esthétique globale pour des graphiques professionnels
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11

print("[+] Bibliothèques importées avec succès et style configuré.")

In [ ]:
# 1. Chargement du fichier CSV généré à l'étape 4
try:
    df = pd.read_csv("anssi_cve_enrichi.csv")
    print("[+] Fichier 'anssi_cve_enrichi.csv' chargé avec succès.")
except FileNotFoundError:
    print("[!] Fichier introuvable. Création d'un jeu de données fictif pour la démonstration...")

    # Génération de données de secours si le CSV n'existe pas localement
    np.random.seed(42)
    n_rows = 150
    editeurs = ['Microsoft', 'Ivanti', 'Cisco', 'Apache', 'Linux', 'Apple', 'Oracle']
    cwes = ['CWE-287', 'CWE-77', 'CWE-89', 'CWE-119', 'CWE-200', 'CWE-79']

    fake_data = {
        'ID ANSSI': [f"CERTFR-2024-ALE-{i:03d}" if i % 5 == 0 else f"CERTFR-2024-AVI-{i:03d}" for i in range(1, n_rows+1)],
        'Titre ANSSI': [f"Vulnérabilité dans {np.random.choice(editeurs)}" for _ in range(n_rows)],
        'Type': ['Alerte' if i % 5 == 0 else 'Avis' for i in range(1, n_rows+1)],
        'Date': pd.date_range(start='2024-01-01', periods=n_rows, freq='D').strftime('%Y-%m-%d'),
        'CVE': [f"CVE-2024-{np.random.randint(1000, 9999)}" for _ in range(n_rows)],
        'CVSS': np.round(np.random.uniform(2.0, 10.0, n_rows), 1),
        'CWE': [np.random.choice(cwes) for _ in range(n_rows)],
        'EPSS': np.round(np.random.beta(0.2, 2, n_rows), 4),
        'Lien': ["https://www.cert.ssi.gouv.fr/" for _ in range(n_rows)],
        'Éditeur': [np.random.choice(editeurs) for _ in range(n_rows)],
        'Produit': ['Software v1' for _ in range(n_rows)]
    }
    df = pd.DataFrame(fake_data)

    # Recréation de la colonne de sévérité
    def get_sev(score):
        if score >= 9.0: return 'Critical'
        elif score >= 7.0: return 'High'
        elif score >= 4.0: return 'Medium'
        else: return 'Low'
    df['Base Severity'] = df['CVSS'].apply(get_sev)

# 2. Alignement du typage de la date (essentiel pour les graphiques temporels)
df['Date'] = pd.to_datetime(df['Date'])

# 3. Affichage des dimensions et des premières lignes
print(f"Dimensions du DataFrame : {df.shape[0]} lignes, {df.shape[1]} colonnes\n")
display(df.head())

In [ ]:
print("--- Structure des données ---")
df.info()

print("\n--- Statistiques des variables numériques ---")
display(df.describe())

print("\n--- Répartition de la sévérité ---")
display(df['Base Severity'].value_counts())

In [ ]:
plt.figure(figsize=(11, 6))
sns.histplot(data=df, x='CVSS', bins=15, kde=True, color='#2b5c8f', edgecolor='black')

# Lignes verticales pour marquer les seuils de gravité majeurs
plt.axvline(4.0, color='orange', linestyle='--', label='Medium (>= 4.0)')
plt.axvline(7.0, color='red', linestyle='--', label='High (>= 7.0)')
plt.axvline(9.0, color='darkred', linestyle='-', linewidth=2, label='Critical (>= 9.0)')

plt.title('Distribution des scores de criticité CVSS', fontsize=14, fontweight='bold')
plt.xlabel('Score CVSS v3', fontsize=12)
plt.ylabel('Nombre de vulnérabilités (CVE)', fontsize=12)
plt.xlim(0, 10)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# On prend le Top 6 pour que le graphique reste lisible
top_cwe = df['CWE'].value_counts().head(6)

plt.figure(figsize=(8, 8))
colors = sns.color_palette('pastel')[0:len(top_cwe)]
plt.pie(top_cwe, labels=top_cwe.index, autopct='%1.1f%%', startangle=140, colors=colors,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.5, 'antialiased': True})

plt.title('Top des catégories de faiblesses les plus fréquentes (CWE)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 5))
sns.kdeplot(data=df, x='EPSS', fill=True, color='#2ca02c', alpha=0.4, linewidth=2)
sns.rugplot(data=df, x='EPSS', color='#1f77b4', alpha=0.6)

plt.title("Distribution de la probabilité d'exploitation réelle (Score EPSS)", fontsize=14, fontweight='bold')
plt.xlabel('Score EPSS (0 à 1)', fontsize=12)
plt.ylabel('Densité', fontsize=12)
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
top_editors = df['Éditeur'].value_counts().head(10)

plt.figure(figsize=(11, 6))
sns.barplot(x=top_editors.values, y=top_editors.index, hue=top_editors.index, palette='viridis', legend=False)

plt.title('Top 10 des éditeurs/constructeurs les plus affectés', fontsize=14, fontweight='bold')
plt.xlabel('Nombre total de CVE recensées', fontsize=12)
plt.ylabel('Nom de l\'éditeur', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 7))

# Définition de l'ordre de sévérité et filtrage des valeurs présentes
severity_order = ['Low', 'Medium', 'High', 'Critical']
present_severities = [s for s in severity_order if s in df['Base Severity'].unique()]

sns.scatterplot(data=df, x='CVSS', y='EPSS', hue='Base Severity', hue_order=present_severities,
                palette={'Low': 'lightblue', 'Medium': 'orange', 'High': 'red', 'Critical': 'purple'},
                s=100, alpha=0.8, edgecolor='black')

plt.title('Relation entre Gravité Théorique (CVSS) et Menace Réelle (EPSS)', fontsize=14, fontweight='bold')
plt.xlabel('Score CVSS (Gravité intrinsèque)', fontsize=12)
plt.ylabel('Score EPSS (Probabilité d\'exploitation active)', fontsize=12)
plt.xlim(0, 10.5)
plt.ylim(-0.05, 1.05)
plt.legend(title='Sévérité CVSS')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

# On isole les données numériques pour éviter les erreurs de corrélation sur des chaînes de caractères
corr_matrix = df[['CVSS', 'EPSS']].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".3f", linewidths=1, linecolor='white')
plt.title('Matrice de corrélation de Pearson (CVSS vs EPSS)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Groupement par date et tri temporel
df_time = df.sort_values('Date').groupby('Date').size().reset_index(name='Nombre')
df_time['Cumul'] = df_time['Nombre'].cumsum()

plt.figure(figsize=(12, 6))
plt.plot(df_time['Date'], df_time['Cumul'], color='#d62728', linewidth=3, marker='o', markersize=4)

plt.title('Évolution cumulative du nombre de vulnérabilités détectées dans le temps', fontsize=14, fontweight='bold')
plt.xlabel('Date de publication du bulletin ANSSI', fontsize=12)
plt.ylabel('Nombre total cumulé de CVE', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# On cible le Top 5 des éditeurs pour ne pas surcharger le graphe
top_5_editors = df['Éditeur'].value_counts().head(5).index
df_filtered = df[df['Éditeur'].isin(top_5_editors)]

plt.figure(figsize=(11, 6))
sns.boxplot(data=df_filtered, x='Éditeur', y='CVSS', hue='Éditeur', palette='Set2', legend=False)
sns.stripplot(data=df_filtered, x='Éditeur', y='CVSS', color='black', alpha=0.3, size=5, jitter=0.2)

plt.title('Dispersion et distribution des scores CVSS pour les éditeurs majeurs', fontsize=14, fontweight='bold')
plt.xlabel('Éditeur', fontsize=12)
plt.ylabel('Distribution des scores CVSS', fontsize=12)
plt.tight_layout()
plt.show()